# Status Type Listing

This notebook lists all distinct `status` values in the CSV, including counts.
It also labels null and blank values explicitly.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Change file name if needed
FILE_PATH = "transaction.csv"
STATUS_COLUMN = "status"

df = pd.read_csv(FILE_PATH)
print(f"Loaded: {FILE_PATH} | shape={df.shape}")
print("Columns:", list(df.columns))

Loaded: transaction.csv | shape=(10103, 6)
Columns: ['status', 'time', 'card_type', 'city', 'amount', 'id']


In [3]:
if STATUS_COLUMN not in df.columns:
    raise ValueError(f"Column '{STATUS_COLUMN}' not found.")

status_clean = df[STATUS_COLUMN].astype("string").str.strip()
status_label = status_clean.fillna("<NULL>").replace("", "<BLANK>")

status_table = (
    status_label
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="row_count")
)

status_table["pct_of_total_rows"] = (status_table["row_count"] / len(df) * 100).round(2)

print(f"Distinct status labels: {len(status_table)}")
display(status_table)

Distinct status labels: 6


,status,row_count,pct_of_total_rows
0,fail,4005,39.64
1,success,3963,39.23
2,failed,1193,11.81
3,Success,498,4.93
4,FAIL,223,2.21
5,succeed,221,2.19


In [7]:
# Time format check: YYYY-MM-DD HH:MM:SS
# Example valid value: 2025-09-07 10:48:00
# valid if matches format and parseable, otherwise invalid

TIME_COLUMN = "time"

if TIME_COLUMN not in df.columns:
    raise ValueError(f"Column '{TIME_COLUMN}' not found.")

time_txt = df[TIME_COLUMN].astype("string").str.strip()

# Match exactly: YYYY-MM-DD HH:MM:SS
fmt_ok = time_txt.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", na=False)
parsed_ok = pd.to_datetime(time_txt, format="%Y-%m-%d %H:%M:%S", errors="coerce").notna()

df_time_check = df[[TIME_COLUMN]].copy()
df_time_check["time_format_status"] = "invalid"
df_time_check.loc[fmt_ok & parsed_ok, "time_format_status"] = "valid"

summary = (
    df_time_check["time_format_status"]
    .value_counts()
    .reindex(["valid", "invalid"], fill_value=0)
    .rename_axis("status")
    .reset_index(name="count")
)

display(summary)

print("Invalid time examples:")
display(df_time_check[df_time_check["time_format_status"] == "invalid"].head(50))

,status,count
0,valid,10000
1,invalid,103


Invalid time examples:


,time,time_format_status
0,23:17 2025-09-08,invalid
1,NaN,invalid
2,07:32 2025-09-20,invalid
3,03-00-2025 09-11,invalid
4,22:27 2025-09-11,invalid
5,02:24 2025-09-04,invalid
6,17:07 2025-09-30,invalid
7,18:12 2025-09-23,invalid
8,02:54 2025-09-08,invalid
9,05:23 2025-09-12,invalid


In [8]:
# List all card_type values (including null/blank)
CARD_TYPE_COLUMN = "card_type"

if CARD_TYPE_COLUMN not in df.columns:
    raise ValueError(f"Column '{CARD_TYPE_COLUMN}' not found.")

card_clean = df[CARD_TYPE_COLUMN].astype("string").str.strip()
card_label = card_clean.fillna("<NULL>").replace("", "<BLANK>")

card_type_table = (
    card_label
    .value_counts(dropna=False)
    .rename_axis("card_type")
    .reset_index(name="row_count")
)

card_type_table["pct_of_total_rows"] = (card_type_table["row_count"] / len(df) * 100).round(2)

print(f"Distinct card_type labels: {len(card_type_table)}")
display(card_type_table)

Distinct card_type labels: 11


,card_type,row_count,pct_of_total_rows
0,Visa,2383,23.59
1,Discover,1663,16.46
2,MasterCard,1461,14.46
3,Amex,1275,12.62
4,MastCard,1010,10.0
5,Vsa,810,8.02
6,Master-Card,601,5.95
7,<NULL>,314,3.11
8,visa,208,2.06
9,Master Card,196,1.94


In [9]:
# List all city values (including null/blank)
CITY_COLUMN = "city"

if CITY_COLUMN not in df.columns:
    raise ValueError(f"Column '{CITY_COLUMN}' not found.")

city_clean = df[CITY_COLUMN].astype("string").str.strip()
city_label = city_clean.fillna("<NULL>").replace("", "<BLANK>")

city_table = (
    city_label
    .value_counts(dropna=False)
    .rename_axis("city")
    .reset_index(name="row_count")
)

city_table["pct_of_total_rows"] = (city_table["row_count"] / len(df) * 100).round(2)

print(f"Distinct city labels: {len(city_table)}")
display(city_table)

Distinct city labels: 14


,city,row_count,pct_of_total_rows
0,Tehran,2279,22.56
1,Tabriz,1381,13.67
2,Isfahan,1084,10.73
3,Mashhad,909,9.0
4,Shiraz,734,7.27
5,Qom,676,6.69
6,Karaj,649,6.42
7,Ahvaz,638,6.31
8,THR,414,4.1
9,TEHRAN,408,4.04


In [12]:
# Amount format check: integer-only or decimal
AMOUNT_COLUMN = "amount"

if AMOUNT_COLUMN not in df.columns:
    raise ValueError(f"Column '{AMOUNT_COLUMN}' not found.")

amount_txt = df[AMOUNT_COLUMN].astype("string").str.strip()
amount_num = pd.to_numeric(amount_txt, errors="coerce")

amount_type = pd.Series("non_numeric", index=df.index, dtype="string")

null_mask = amount_txt.isna() | (amount_txt == "")
amount_type[null_mask] = "NULL"

numeric_mask = amount_num.notna() & ~null_mask
amount_type[numeric_mask & ((amount_num % 1) == 0)] = "integer"
amount_type[numeric_mask & ((amount_num % 1) != 0)] = "decimal"

amount_check = df[[AMOUNT_COLUMN]].copy()
amount_check["amount_format_type"] = amount_type

summary = (
    amount_check["amount_format_type"]
    .value_counts(dropna=False)
    .rename_axis("amount_format_type")
    .reset_index(name="row_count")
)
summary["pct_of_total_rows"] = (summary["row_count"] / len(df) * 100).round(2)

display(summary)

print("Sample rows by amount_format_type:")
display(amount_check.groupby("amount_format_type", dropna=False).head(20))

,amount_format_type,row_count,pct_of_total_rows
0,integer,9791,96.91
1,decimal,309,3.06
2,NULL,2,0.02
3,non_numeric,1,0.01


Sample rows by amount_format_type:


,amount,amount_format_type
0,-5000,integer
1,63489,integer
2,1502601,integer
3,9999999999,integer
4,452545,integer
5,9999999999,integer
6,1158713,integer
7,1387769,integer
8,583952,integer
9,1472823,integer
